# 50 ? Phase 6: Abstractive Legal Summarization

Generate **one summary per document** from your Phase-1 models. The official schema is
`{"id": "...", "document": "...", "summary": "..."}` for training; test references remain private.
There is **no public dev.json**. We create a deterministic holdout from train.json, grouping duplicate
normalized documents together. All experiments and the KD teacher must share the same holdout.

Brief ?12: use all four backbones, LoRMA is compulsory, at least three adaptation techniques plus a
smaller-student knowledge-distillation experiment. The table includes **five** runs: A/LoRA, B/LoRMA,
C/LoRA, D/KD, plus A/QLoRA. This covers all four backbones and the three listed PEFT techniques
LoRA/LoRMA/QLoRA plus KD. Choose your own settings and compare actual results; these are unrun templates.

Primary metric: **BERTScore F1**. Secondary: **ROUGE-1 F1, ROUGE-L F1**. The notebook selects by
BERTScore by default; the evaluation encoder is used only for scoring, never adaptation or teaching.
Local BERTScore is approximate unless its encoder/settings match the official evaluator.

The bundled brief places Phase 6 on October 27?November 2, 2026. Runs persist in
`MyDrive/cs781/summ_runs/`. Review `CONTEXT.md` for data assumptions and limitations.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# Install before importing torch/backbones. Restart the runtime if pip requests it.
%pip -q install numpy pandas tokenizers ijson "huggingface_hub[hf_xet]" rouge-score bert-score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, glob, zipfile, tempfile, importlib, shutil, time, json
from pathlib import Path
MYDRIVE = Path('/content/drive/MyDrive')
DRIVE_ROOT = MYDRIVE / 'cs781'
RUNS_DIR = DRIVE_ROOT / 'runs'
OUT_DIR = DRIVE_ROOT / 'summ_runs'
P6 = Path('/content/p6')
P6.mkdir(exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional explicit folder or archive; leave None to find an updated Phase-6 code copy.
CODE_SOURCE = None
levels = ['', '*/', '*/*/', '*/*/*/']
if CODE_SOURCE is None:
    folders = [Path(h).parents[2] for level in levels
               for h in glob.glob(str(MYDRIVE / f'{level}lm_lib/summ/trainer.py'))]
    archives = [Path(h) for level in levels for h in glob.glob(str(MYDRIVE / f'{level}cs781_code*.zip'))]
    valid_archives = []
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            if 'lm_lib/summ/trainer.py' in z.namelist():
                valid_archives.append(archive)
    candidates = [(p, (p / 'lm_lib/summ/trainer.py').stat().st_mtime) for p in folders]
    candidates += [(p, p.stat().st_mtime) for p in valid_archives]
    if not candidates:
        raise FileNotFoundError('Upload updated lm_lib/, models/, adapters/ folders, or rebuild and upload cs781_code.zip.')
    CODE_SOURCE = max(candidates, key=lambda item: item[1])[0]
CODE_SOURCE = Path(CODE_SOURCE)
if CODE_SOURCE.is_file():
    CODE_DIR = Path(tempfile.mkdtemp(prefix='cs781_phase6_', dir='/content'))
    with zipfile.ZipFile(CODE_SOURCE) as z:
        z.extractall(CODE_DIR)
else:
    CODE_DIR = CODE_SOURCE
assert (CODE_DIR / 'lm_lib/summ/trainer.py').is_file(), 'Selected code copy lacks Phase 6'
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
# Drop project modules cached from a previous phase/archive so imports use this code copy.
for name in list(sys.modules):
    if name in ('lm_lib', 'models', 'adapters') or name.startswith(('lm_lib.', 'models.', 'adapters.')):
        del sys.modules[name]
# Re-run the later import/configuration cells after re-running setup.
print('Code:', CODE_SOURCE, '\nRuns:', OUT_DIR)

In [ ]:
from huggingface_hub import hf_hub_download, login

def get_hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN'].strip()
    token_file = CODE_DIR / 'hf_token.txt'
    if token_file.is_file():
        return token_file.read_text().strip()
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Set HF_TOKEN in Colab Secrets or the environment, or provide hf_token.txt beside the code.') from exc

HF_TOKEN = get_hf_token()
login(token=HF_TOKEN, add_to_git_credential=False)
# These paths and the record schema are specified in the bundled brief ?12.3.
cache_dir = DRIVE_ROOT / 'data' / 'phase6'
cache_dir.mkdir(parents=True, exist_ok=True)
for relative in ['Train/train.json', 'Test/test.json']:
    name = Path(relative).name
    cached = cache_dir / name
    if not cached.is_file():
        downloaded = hf_hub_download('Exploration-Lab/CS-781-Capstone', f'Phase_6_Summ/{relative}',
                                     repo_type='dataset', token=HF_TOKEN)
        shutil.copyfile(downloaded, cached)
    shutil.copyfile(cached, P6 / name)
    print(name, f'{(P6 / name).stat().st_size / 1e6:.1f} MB')

In [ ]:
PHASE1 = {
    'A': RUNS_DIR / 'A_qwen_dense_full',
    'B': RUNS_DIR / 'B_encdec',
    'C': RUNS_DIR / 'C_gdn_hybrid',
    'D': RUNS_DIR / 'D_mla_moe',
}
for key, path in PHASE1.items():
    print(key, path, '| checkpoints:', sorted(p.name for p in path.glob('*.pt')))

# Select one run at a time; repeat from this cell to compare techniques/backbones.
# A_lora must finish before D_kd. A_qlora is an additional run covering a third listed PEFT technique.
EXPERIMENTS = {
    'A_lora': dict(arch='A', method='lora', method_kwargs={'r': 8, 'lora_alpha': 16},
                   run_name='A_lora__firstlast640_256'),
    'B_lorma': dict(arch='B', method='lorma', method_kwargs={'r': 8, 'lorma_alpha': 16},
                    run_name='B_lorma__firstlast640_256'),
    'C_lora': dict(arch='C', method='lora',
                   method_kwargs={'r': 8, 'lora_alpha': 16, 'target_modules': ['mlp.', 'attn.']},
                   run_name='C_lora__firstlast640_256'),
    'D_kd': dict(arch='D', method='lora', method_kwargs={'r': 8, 'lora_alpha': 16},
                 run_name='D_kd_from_A__firstlast640_256',
                 kd_teacher=str(OUT_DIR / 'A_lora__firstlast640_256')),
    'A_qlora': dict(arch='A', method='qlora', method_kwargs={'r': 8, 'lora_alpha': 16},
                    run_name='A_qlora__firstlast640_256'),
}
EXPERIMENT = 'A_qlora'
selected = EXPERIMENTS[EXPERIMENT].copy()
ARCH = selected.pop('arch')
if ARCH == 'C':
    %pip -q install flash-linear-attention
if selected['method'] == 'qlora':
    %pip -q install bitsandbytes
# Existing architecture/adapter files are reused unchanged.
# C targets the repository MLP/attention projections, leaving FLA mixer internals untouched.


In [ ]:
import numpy as np
import pandas as pd
from lm_lib.summ import data as sd
all_docs = sd.load_split(P6 / 'train.json')
test_docs = sd.load_split(P6 / 'test.json', labeled=False)
assert not ({d['id'] for d in all_docs} & {d['id'] for d in test_docs}), 'Train/test IDs overlap'
print('Public training docs:', len(all_docs), '| test docs:', len(test_docs))
for field in ['document', 'summary']:
    lengths = [len(d[field].split()) for d in all_docs]
    print(field, 'word-length percentiles [50, 90, 99, max]:', np.percentile(lengths, [50, 90, 99, 100]).astype(int))
print('Example schema:', {key: type(value).__name__ for key, value in all_docs[0].items()})
print('Training fingerprint:', sd.fingerprint(all_docs))
# Do not inspect or use test references. The test loader only reads ID and document.
del all_docs, test_docs

**Review these settings before training.** Optimization values inherit earlier-phase starting values;
they are not tuned recommendations. `source_tokens` includes BOS and document/summary markers.
`target_tokens` includes EOT. Training references longer than this budget are truncated, then EOT is appended.
`max_new_tokens` must not exceed that trained target budget. Source + target must fit a decoder's context;
B separately checks its pretrained encoder and decoder budgets.

`first_last` selects exact token spans from both ends; `first` and `last` are alternatives. It tokenizes
all source text before selecting and does not claim full-document or hierarchical coverage. Choose the
representation and compare information coverage yourself. References are never used for source selection.

One document per forward avoids padding leakage in the custom models, which do not accept attention masks.
`grad_accum` controls the effective batch. Generation reuses the existing
model caches where available, allows EOT and never enforces Phase 1's 128-word continuation rule.

Use the same split seed/fraction and eval subset across runs. `n_val_generate=0` evaluates all held-out docs;
a nonzero value limits both generation metrics and validation NLL to a saved reproducible subset. A subset
score must not be described as a score on the full validation set.

BERTScore downloads a scoring encoder used only for evaluation.
If intentionally selecting by `rougeL_f1`, record that choice and still report BERTScore separately below.
The official scorer settings are not specified in the brief, so local scores are estimates.

In [ ]:
from lm_lib.summ.trainer import SummConfig, train_summ, predict_test
cfg = SummConfig(
    phase1_run=str(PHASE1[ARCH]), phase1_ckpt='latest', **selected,
    data_dir=str(P6), out_dir=str(OUT_DIR),
    val_fraction=0.05, split_seed=1337, n_train=0, n_val_generate=32,
    source_tokens=640, target_tokens=256, strategy='first_last', normalize_text=True,
    grad_accum=8, grad_checkpoint=False,
    epochs=1, lr=5e-5, weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    eval_every=0, selection_metric='bertscore_f1', bert_model='roberta-large', bert_device='cpu',
    max_new_tokens=256, min_new_tokens=16,
    temperature=0.0, top_k=0, top_p=1.0, repetition_penalty=1.0, no_repeat_ngram_size=0,
    kd_alpha=0.5, stop_at='', seed=1337,
)
run_dir = OUT_DIR / cfg.run_name
assert not run_dir.exists() or not any(run_dir.iterdir()), 'Use a fresh run_name; training does not resume optimizer state'
for name in ['latest.pt', 'tokenizer.json']:
    assert (PHASE1[ARCH] / name).is_file(), f'Missing Phase-1 asset: {PHASE1[ARCH] / name}'
if cfg.kd_teacher:
    assert (Path(cfg.kd_teacher) / 'best.pt').is_file(), 'Train A_lora teacher first'
print(cfg)

In [ ]:
# Preflight actual checkpoint, objective alignment and decoder/adapter gradient flow.
import gc, torch
from lm_lib.summ.trainer import build_model, summary_loss
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tok, model_type, report = build_model(cfg, DEVICE)
all_docs = sd.load_split(P6 / 'train.json')
train_docs, val_docs = sd.split_train_val(all_docs, cfg.val_fraction, cfg.split_seed)
print(DEVICE, '| model type:', model_type, '| parameters:', report)
print('Train pool:', len(train_docs), '| held-out:', len(val_docs))
example = sd.prepare(train_docs[0], tok, cfg, model_type)
loss, ntokens = summary_loss(model, example, model_type, DEVICE)
assert torch.isfinite(loss) and ntokens > 0
(loss / ntokens).backward()
assert any(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0
           for p in model.parameters() if p.requires_grad), 'No useful trainable gradients'
print('Source/input tokens:', len(example.get('src', example['x'])), '| summary loss tokens:', ntokens)
print('Preflight mean CE:', float(loss.detach()) / ntokens)
# B uses encoder AND decoder; unlike classification, its decoder must train here.
if model_type == 'encdec':
    assert any(p.requires_grad and p.grad is not None for p in model.decoder.parameters()), 'Decoder received no gradients'
del model, tok, all_docs, train_docs, val_docs, example, loss
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

For **sequence KD**, the teacher is a completed Phase-6 run that started from your own Phase-1 weights.
The trainer verifies the same dataset fingerprint and held-out IDs, and checks that the student's original
backbone has fewer parameters. It generates teacher summaries **only for the selected student training IDs**
and records teacher checkpoint provenance in `kd_targets.json`. Original reference and teacher-summary losses
are mixed using `kd_alpha`. Text targets permit different student/teacher tokenizers and architectures.

If your actual D checkpoint is not smaller than A, choose a genuinely smaller Phase-1 student. The KD
check will reject it rather than label an invalid size comparison as the required experiment.

`stop_at` checks between accumulation groups; an evaluation/checkpoint finishes afterwards. Existing run
folders are protected, and there is no optimizer-resume support. PEFT checkpoints contain adapters and
mutable buffers; preserve the original Phase-1 checkpoint/tokenizer alongside them.

In [ ]:
results = train_summ(cfg)
print('Run:', run_dir)
print('Best held-out metrics:', results['best_val_metrics'])
print('Optimizer steps:', results['optimizer_steps'], '| stopped early:', results['stopped_early'])

In [ ]:
# Inspect generated validation summaries against references. Write your own factuality analysis.
manifest = json.loads((run_dir / 'split.json').read_text())
val_predictions = json.loads((run_dir / 'val_predictions.json').read_text())
reference_docs = {d['id']: d for d in sd.load_split(P6 / 'train.json') if d['id'] in set(manifest['eval_ids'])}
for ident in list(val_predictions)[:3]:
    print('ID:', ident)
    print('REFERENCE:', reference_docs[ident]['summary'])
    print('GENERATED:', val_predictions[ident], '\n')
assert set(val_predictions) == set(manifest['eval_ids'])
# Report all three metrics for the selected checkpoint, including when selection used ROUGE-L.
from lm_lib.summ.metrics import score_summaries
val_metrics = score_summaries(val_predictions, {i: reference_docs[i]['summary'] for i in val_predictions},
                              bertscore=True, bert_model=cfg.bert_model, bert_device=cfg.bert_device)
(run_dir / 'reported_val_metrics.json').write_text(json.dumps(val_metrics, indent=2))
print(val_metrics)
del reference_docs

In [ ]:
# Reload best weights and saved generation settings, then generate one summary per test ID.
zip_path, predictions = predict_test(str(run_dir), data_dir=str(P6))
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist() == ['prediction.json']
    submission = json.loads(z.read('prediction.json'))
assert len(submission) == len({r['id'] for r in submission})
assert all(isinstance(r['summary'], str) and r['summary'].strip() for r in submission)
print('Saved:', zip_path)
print('Preview:', submission[:2])
# Optional:
# from google.colab import files
# files.download(zip_path)

In [ ]:
# Evidence table: do not compare runs evaluated on different holdouts/subsets as matched experiments.
rows = []
for path in OUT_DIR.glob('*/results.json'):
    result = json.loads(path.read_text())
    saved_cfg = json.loads((path.parent / 'config.json').read_text())
    split = json.loads((path.parent / 'split.json').read_text())
    metric_path = path.parent / 'reported_val_metrics.json'
    metrics = json.loads(metric_path.read_text()) if metric_path.exists() else result['best_val_metrics']
    rows.append(dict(run=result['run'], phase1=Path(result['phase1_run']).name, method=result['method'],
                     kd=bool(result['kd_teacher']), selection_metric=result['selection_metric'],
                     bertscore_f1=metrics.get('bertscore_f1'), rouge1_f1=metrics['rouge1_f1'],
                     rougeL_f1=metrics['rougeL_f1'], train_docs=result['train_docs'], eval_docs=result['eval_docs'],
                     base_params=result['base_params'], trainable_params=result['trainable_params'],
                     trainable_pct=result['trainable_pct'],
                     stopped_early=result['stopped_early'],
                     split_key=(split['data_sha256'], tuple(sorted(split['val_ids'])), tuple(sorted(split['eval_ids'])))))
comparison = pd.DataFrame(rows)
if not comparison.empty:
    print('Identical dataset/holdout/eval IDs:', comparison['split_key'].nunique() == 1)
    display(comparison.drop(columns='split_key').sort_values('bertscore_f1', ascending=False))
    comparison.drop(columns='split_key').to_csv(OUT_DIR / 'comparison.csv', index=False)
    print('Distinct Phase-1 runs:', comparison.phase1.nunique(), '| methods:', sorted(comparison.method.unique()),
          '| LoRMA:', bool((comparison.method == 'lorma').any()), '| KD:', bool(comparison.kd.any()))
print('Export submissions for all four backbones. Templates/counts alone do not establish completed requirements.')

Retain `config.json`, `split.json`, `tokenizer.json`, `best.pt`, `log.csv`, validation predictions/metrics,
`results.json`, KD targets where applicable, and each submission's inference settings. PEFT runs also need
the unchanged original Phase-1 checkpoint; its SHA-256 and tokenizer are checked on reload.

Complete your report yourself: compare the adaptation methods and KD with an undistilled student; inspect
facts, legal reasoning, coverage and repetition; compare source selection and output lengths; compare trainable parameters. Optimization/decoding numbers here are starting values, not tuned
claims. Cite the existing methods, disclose ancillary assistance and comply with brief ?17.5.

Local tests use tiny A/B configurations. QLoRA follows
[bitsandbytes Linear4bit](https://huggingface.co/docs/bitsandbytes/reference/nn/linear4bit), quantizes only
LoRA base projections, and keeps tied embeddings/output in full precision. It is enabled only on A.
BERTScore settings are local approximations until matched against the official evaluator.